# Beginner GeoPrompt Workflow

This walkthrough uses the checked-in sample data and verifies a few core frame operations:

- load mixed geometry features
- inspect schema and bounds
- run a small spatial filter
- export a real artifact to the outputs folder
- render a compact visual summary

When this notebook is committed with cached outputs, GitHub will display the results directly in the notebook viewer.

In [1]:
from pathlib import Path
import geoprompt as gp


def find_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "sample_features.json").exists():
            return candidate
    return Path.cwd()


root = find_root()
output_dir = root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

frame = gp.read_data(root / "data" / "sample_features.json")
print("Loaded rows:", len(frame))
print(frame.summary())
print(frame.head(3))

Loaded rows: 6
{'row_count': 6, 'column_count': 6, 'columns': ['site_id', 'name', 'geometry', 'demand_index', 'capacity_index', 'priority_index'], 'crs': None, 'geometry_types': ['LineString', 'Point', 'Polygon'], 'bounds': {'min_x': -112.09, 'min_y': 40.55, 'max_x': -111.74, 'max_y': 40.78}, 'column_stats': [{'column': 'site_id', 'dtype': 'string', 'null_count': 0, 'unique_count': 6}, {'column': 'name', 'dtype': 'string', 'null_count': 0, 'unique_count': 6}, {'column': 'demand_index', 'dtype': 'numeric', 'null_count': 0, 'unique_count': 6, 'min': 0.69, 'max': 0.94, 'mean': 0.835}, {'column': 'capacity_index', 'dtype': 'numeric', 'null_count': 0, 'unique_count': 6, 'min': 0.71, 'max': 0.95, 'mean': 0.8300000000000001}, {'column': 'priority_index', 'dtype': 'numeric', 'null_count': 0, 'unique_count': 6, 'min': 0.86, 'max': 1.22, 'mean': 1.0683333333333334}]}
[{'site_id': 'north-hub-point', 'name': 'North Intake Hub', 'geometry': {'type': 'Point', 'coordinates': (-111.92, 40.78)}, 'deman

In [3]:
window = frame.query_bounds(-112.0, 40.6, -111.8, 40.8)
written = output_dir / "notebook-beginner.geojson"
gp.write_data(written, window)
print("Filtered rows:", len(window))
print("Geometry areas:", frame.geom.area())
print("Wrote:", written)

Filtered rows: 6
Geometry areas: [0.0, 0.0, 0.0, 0.0, 0.008099999999103602, 0.008999999999105057]
Wrote: d:\Github\geoprompt\outputs\notebook-beginner.geojson


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

rows = frame.to_records() if hasattr(frame, 'to_records') else list(frame)
labels = [row['site_id'].replace('-point', '').replace('-line', '').replace('-zone', '') for row in rows]
demand = [row.get('demand_index', 0.0) for row in rows]
capacity = [row.get('capacity_index', 0.0) for row in rows]

# Modern style
plt.rcParams.update({
    'figure.facecolor': '#f8fafc',
    'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#e2e8f0',
    'axes.grid': True,
    'grid.color': '#f1f5f9',
    'grid.linewidth': 0.8,
    'font.family': 'sans-serif',
    'font.size': 10,
})
fig, ax = plt.subplots(figsize=(9, 4))

x = range(len(labels))
w = 0.38
bars = ax.bar([i - w/2 for i in x], demand, w, label='Demand', color='#3b82f6', edgecolor='#2563eb', linewidth=0.5, zorder=3)
ax.bar([i + w/2 for i in x], capacity, w, label='Capacity', color='#22c55e', edgecolor='#16a34a', linewidth=0.5, zorder=3)

# Value labels on bars
for bar, val in zip(bars, demand):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.2f}',
            ha='center', va='bottom', fontsize=8, color='#475569', fontweight='500')

ax.set_xticks(list(x))
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Index value', fontsize=10, color='#334155')
ax.set_title('Sample Feature — Demand vs Capacity', fontsize=14, fontweight='bold', color='#0f172a', pad=14)
ax.legend(frameon=True, fancybox=True, shadow=False, edgecolor='#e2e8f0', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(colors='#64748b')
fig.tight_layout()

image_path = output_dir / 'notebook-beginner-summary.png'
fig.savefig(image_path, dpi=180, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close(fig)
display(Image(filename=str(image_path)))
print('Saved:', image_path)